# Train / Validation / Test Splits

CBIS-DDSM provides an official train/test partition, but no validation. Validation is required for hyperparameter tuning.
The validation set would be carved from the training set.

In [1]:
import subprocess
from pathlib import Path

import pandas as pd

CBIS_DDSM = Path("../data/cbis-ddsm")
SPLITS_DIR = Path("../data/cbis-ddsm/training")

## The Official Partition

In [2]:
sizes = {}
for name in ["mass_case_description_train_set.csv",
             "mass_case_description_test_set.csv",
             "calc_case_description_train_set.csv",
             "calc_case_description_test_set.csv"]:
    p = CBIS_DDSM / name
    sizes[name] = len(pd.read_csv(p))

for name, s in sizes.items():
    print(f"{name}\t{s}")

mass_case_description_train_set.csv	1318
mass_case_description_test_set.csv	378
calc_case_description_train_set.csv	1546
calc_case_description_test_set.csv	326


## Patient-Disjoint Validation Carving

Taking 10% of the training fold for validation, using `StratifiedGroupKFold`, grouped on `paient_id`.

In [3]:

train_csv = SPLITS_DIR / "train.csv"
val_csv = SPLITS_DIR / "val.csv"
test_csv = SPLITS_DIR / "test.csv"

if not train_csv.exists() or not val_csv.exists():
    subprocess.run(["python3", "-m", "src.data.splits"], check=True)

train_df = pd.read_csv(train_csv)
print(f"Train size: {len(train_df)}")
print(f"Train (positive) size: {len(train_df[train_df['label'] == 1])}")
print(f"Train (negative) size: {len(train_df[train_df['label'] == 0])}\n")

val_df = pd.read_csv(val_csv)
print(f"Validation size: {len(val_df)}")
print(f"Validation (positive) size: {len(val_df[val_df['label'] == 1])}")
print(f"Validation (negative) size: {len(val_df[val_df['label'] == 0])}")

test_df = pd.read_csv(test_csv)
print(f"\nTest size: {len(test_df)}")
print(f"Test (positive) size: {len(test_df[test_df['label'] == 1])}")
print(f"Test (negative) size: {len(test_df[test_df['label'] == 0])}")
display(train_df.head())

Train size: 2576
Train (positive) size: 1062
Train (negative) size: 1514

Validation size: 288
Validation (positive) size: 119
Validation (negative) size: 169

Test size: 704
Test (positive) size: 276
Test (negative) size: 428


,image_id,patient_id,dataset,pathology,label,birads_density,mass_or_calc,subtlety,roi_mask_id
0,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,P_00004,cbis_ddsm,BENIGN,0,3,mass,3,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...
1,Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.959...,P_00004,cbis_ddsm,BENIGN,0,3,mass,3,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...
2,Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.95...,P_00004,cbis_ddsm,BENIGN,0,3,mass,5,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....
3,Mass-Training_P_00009_RIGHT_CC/1.3.6.1.4.1.959...,P_00009,cbis_ddsm,MALIGNANT,1,3,mass,4,Mass-Training_P_00009_RIGHT_CC_1/1.3.6.1.4.1.9...
4,Mass-Training_P_00009_RIGHT_MLO/1.3.6.1.4.1.95...,P_00009,cbis_ddsm,MALIGNANT,1,3,mass,4,Mass-Training_P_00009_RIGHT_MLO_1/1.3.6.1.4.1....
